# Lezione 01 — Pensare in modo statistico

**Corso:** Statistica Applicata con Python · **Blocco 1:** Descrivere i dati
**Tempo stimato:** 90 minuti · **Capitolo del manuale:** `manual/01_pensar_estadisticamente_IT.md`

## Cosa vedrai in questa lezione

- La differenza tra **popolazione, campione, parametro e statistica campionaria**, con dati che conosciamo per intero.
- Perché due campioni della stessa popolazione danno risultati diversi (**variabilità campionaria**) e come dipende dalla dimensione del campione.
- Come distinguere il **segnale** dal **rumore**.
- Perché una **distorsione (bias) di campionamento** non sparisce nemmeno se hai moltissimi dati.

## Requisiti

- Corso *Python per la Data Science* (NumPy e Pandas di base).
- Librerie: `numpy`, `pandas`, `matplotlib` (sono già installate in Google Colab).

> **Come usare questo notebook:** esegui le celle in ordine. Prima di ogni esperimento, scrivi (o pensa) cosa ti aspetti che succeda; poi confrontalo con quello che vedi.

---
## Cella 0 — Setup

Importiamo le librerie e fissiamo il **seme** del generatore casuale. Con lo stesso seme, ogni studente ottiene esattamente gli stessi numeri.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)          # seme fisso: tutti otteniamo gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerie caricate correttamente.")

---
## 1. Una popolazione che conosciamo per intero

Nella vita reale non conosci la popolazione. Qui la **generiamo noi**, quindi conosciamo la verità e potremo verificare quanto sbaglia ogni campione.

Simuliamo gli stipendi mensili (in €) di 100 000 dipendenti. È una distribuzione asimmetrica, come lo sono di solito gli stipendi reali. *(Dati sintetici: esempio illustrativo.)*

In [ ]:
N = 100_000
poblacion = rng.lognormal(mean=7.7, sigma=0.4, size=N)    # stipendi mensili sintetici (€)

mu = poblacion.mean()        # PARAMETRO: la media di TUTTA la popolazione
sigma = poblacion.std()      # PARAMETRO: la sua deviazione standard
print(f"Parametri della popolazione:  μ = {mu:,.0f} €    σ = {sigma:,.0f} €")

plt.hist(poblacion, bins=80, color="steelblue", edgecolor="white")
plt.axvline(mu, color="crimson", linestyle="--", label=f"μ = {mu:,.0f} €")
plt.title("Popolazione completa: 100 000 stipendi")
plt.xlabel("Stipendio mensile (€)"); plt.ylabel("Dipendenti"); plt.legend()
plt.show()

---
## 2. Prelevare un campione: la statistica campionaria rispetto al parametro

Ora facciamo quello che faremmo nella realtà: chiediamo solo a 200 dipendenti scelti a caso e calcoliamo la loro media. Quel numero è una **statistica campionaria**. Poi confrontiamo tre campioni diversi della stessa popolazione.

In [ ]:
n = 200
filas = []
for nombre in ["A", "B", "C"]:
    muestra = rng.choice(poblacion, size=n, replace=False)     # campione casuale senza reinserimento
    filas.append({"campione": nombre, "x̄ (statistica)": muestra.mean(), "errore x̄ − μ": muestra.mean() - mu})

tabla = pd.DataFrame(filas).set_index("campione").round(1)
print(f"Parametro reale: μ = {mu:,.1f} €")
tabla

**Osserva:** le tre medie sono diverse tra loro e nessuna è esattamente μ. Nessuna è "sbagliata": è quello che ci si aspetta.

**Domanda per te:** se ripetessi questo esperimento 1 000 volte, come pensi che si distribuirebbero le medie? Attorno a quale valore? Quanto sarebbe ampio il ventaglio? Annotalo prima di andare avanti.

---
## 3. Esperimento guidato: la distribuzione delle medie campionarie

Ripetiamo il campionamento 2 000 volte (sempre con n = 200) e salviamo ogni media. Così non vediamo un solo risultato, ma **tutti i risultati possibili**.

In [ ]:
REPS = 2_000
n = 200

# ogni riga è un campione di dimensione n; la media per righe ci dà 2 000 statistiche campionarie
muestras = rng.choice(poblacion, size=(REPS, n))          # con reinserimento: approssimazione valida (n << N)
medias = muestras.mean(axis=1)

print(f"Media delle {REPS} medie       : {medias.mean():,.1f} €   (μ = {mu:,.1f} €)")
print(f"Deviazione delle medie        : {medias.std():,.1f} €   ← quanto oscilla una statistica campionaria")
print(f"Intervallo osservato          : da {medias.min():,.0f} € a {medias.max():,.0f} €")

plt.hist(medias, bins=50, color="darkorange", edgecolor="white")
plt.axvline(mu, color="crimson", linestyle="--", label="μ reale")
plt.title(f"Distribuzione di {REPS} medie campionarie (n = {n})")
plt.xlabel("Media del campione (€)"); plt.ylabel("Frequenza"); plt.legend()
plt.show()

**Cosa osservi?** Annota le tue conclusioni:

- Le medie si distribuiscono attorno a μ, o attorno a un altro valore?
- La forma dell'istogramma delle medie assomiglia a quella della popolazione (asimmetrica) o è diversa?
- Se nella vita reale avessi *un solo* campione, di quanto potresti sbagliarti al massimo, più o meno?

*(La tua risposta qui.)*

### Effetto della dimensione del campione

Ripetiamo l'esperimento con campioni di dimensione diversa e misuriamo quanto oscillano le medie. Confrontiamo con il valore **σ/√n**, che vedremo in dettaglio nella lezione 6.

In [ ]:
filas = []
for n in [10, 50, 200, 1_000, 5_000]:
    medias_n = rng.choice(poblacion, size=(REPS, n)).mean(axis=1)
    filas.append({"n": n,
                  "oscillazione osservata": medias_n.std(),
                  "σ/√n (teorico)": sigma / np.sqrt(n)})

efecto_n = pd.DataFrame(filas).set_index("n").round(1)
display(efecto_n)

plt.loglog(efecto_n.index, efecto_n["oscillazione osservata"], "o-", label="osservata")
plt.loglog(efecto_n.index, efecto_n["σ/√n (teorico)"], "--", label="σ/√n")
plt.title("Quanto oscilla la media campionaria in base alla dimensione del campione")
plt.xlabel("Dimensione del campione n"); plt.ylabel("Deviazione delle medie (€)"); plt.legend()
plt.show()

**Cosa osservi?**

- Passando da n = 50 a n = 200 (quattro volte più dati), l'oscillazione si dimezza? E da n = 200 a n = 5 000?
- Conviene sempre raccogliere più dati? Quanto costa dimezzare il rumore?

*(La tua risposta qui.)*

---
## 4. Tipi di variabili: non ogni numero è numerico

`pandas` decide il tipo in base a come arrivano i dati, non in base al loro significato. Vediamo cosa succede con un codice postale.

In [ ]:
df = pd.DataFrame({
    "cp": [28001, 15702, 8001, 41001],                    # codice postale (il 08001 ha perso il suo zero!)
    "hijos": [0, 2, 1, 3],                                # numerica discreta
    "satisfaccion": ["alta", "bassa", "media", "alta"],   # categorica ordinale
    "sueldo": [2100.5, 1850.0, 2500.75, 3100.2],          # numerica continua
})
print(df.dtypes, end="\n\n")
print("Media dei codici postali (non ha alcun senso!):", df["cp"].mean())

# Diciamo a pandas cos'è davvero ogni colonna
df["cp"] = df["cp"].astype(str).str.zfill(5).astype("category")
df["satisfaccion"] = pd.Categorical(df["satisfaccion"], categories=["bassa", "media", "alta"], ordered=True)
print(df.dtypes)
df

**Osserva:** prima di riassumere qualsiasi colonna bisogna decidere di che tipo è. I codici postali sono *categorie* che per caso si scrivono con delle cifre, e la soddisfazione è *ordinale*: ha un ordine (`bassa < media < alta`), ma non possiamo dire che la distanza tra "bassa" e "media" sia la stessa che c'è tra "media" e "alta".

---
## 5. Segnale e rumore

Un'azienda prova due versioni di una pagina web. I tempi di acquisto (minuti) hanno una media di 3,2 e una deviazione standard di 0,8. Useremo campioni di 50 utenti per versione.

**Scenario 1: non c'è nessuna differenza reale.** Le due versioni hanno la stessa distribuzione. Anche così, di quanto possono differire le medie di due campioni?

In [ ]:
REPS = 5_000
n = 50
a = rng.normal(3.2, 0.8, size=(REPS, n)).mean(axis=1)     # versione A
b = rng.normal(3.2, 0.8, size=(REPS, n)).mean(axis=1)     # versione B: STESSA distribuzione
dif_sin_efecto = b - a

print(f"Differenza tipica tra le medie quando NON c'è differenza reale: ±{dif_sin_efecto.std():.2f} min")
print(f"Proporzione di volte in cui |B − A| ≥ 0,20 min per puro caso : {(np.abs(dif_sin_efecto) >= 0.20).mean():.1%}")

plt.hist(dif_sin_efecto, bins=60, color="slategray", edgecolor="white")
plt.axvline(0, color="black")
plt.title("Differenza tra le medie B − A quando non c'è nessuna differenza reale")
plt.xlabel("B − A (minuti)"); plt.ylabel("Frequenza")
plt.show()

**Osserva:** anche se non c'è alcun effetto, le differenze tra le medie oscillano attorno a 0 e, con una certa frequenza, arrivano a 0,2 minuti o più. Vedere una differenza di 0,2 min in *un* campione non dimostra nulla da sola.

**Scenario 2: B è davvero più veloce di 0,2 minuti.**

In [ ]:
b_real = rng.normal(3.0, 0.8, size=(REPS, n)).mean(axis=1)      # B con un miglioramento reale di 0,2 min
dif_con_efecto = b_real - a

print(f"Differenza media osservata (dovrebbe essere vicina a −0,20): {dif_con_efecto.mean():+.3f} min")
print(f"Proporzione di volte in cui B sembra PIÙ LENTA di A (B − A > 0), pur essendo realmente più veloce: {(dif_con_efecto > 0).mean():.1%}")

plt.hist(dif_sin_efecto, bins=60, alpha=0.6, label="nessun effetto reale", color="slategray")
plt.hist(dif_con_efecto, bins=60, alpha=0.6, label="B più veloce di 0,2 min", color="seagreen")
plt.axvline(0, color="black")
plt.title("Segnale e rumore: le due distribuzioni si sovrappongono")
plt.xlabel("B − A (minuti)"); plt.legend()
plt.show()

**Cosa osservi?** Le due distribuzioni **si sovrappongono**. Con campioni di 50 utenti, un miglioramento reale di 0,2 min viene spesso confuso con il rumore: può persino capitare che B *sembri* peggiore. Distinguere il segnale dal rumore è esattamente l'obiettivo degli intervalli di confidenza e dei test di ipotesi (lezioni da 6 a 8).

---
## 6. Distorsione (bias) di campionamento: più dati non la risolvono

Un negozio vuole conoscere la soddisfazione media (da 0 a 10) dei suoi 100 000 clienti. Ma al sondaggio rispondono solo i clienti più soddisfatti: la probabilità di rispondere **cresce con la soddisfazione**. È un caso di **autoselezione**. *(Dati sintetici: esempio illustrativo.)*

In [ ]:
N = 100_000
satisfaccion = np.clip(rng.normal(6.0, 2.0, size=N), 0, 10)
p_responder = 1 / (1 + np.exp(-(satisfaccion - 7)))                   # i più contenti rispondono di più
responde = rng.random(N) < p_responder
respondientes = satisfaccion[responde]

mu_sat = satisfaccion.mean()
print(f"Soddisfazione media REALE della popolazione   : {mu_sat:.2f}")
print(f"Clienti che rispondono al sondaggio           : {responde.sum():,} su {N:,}")
print(f"Soddisfazione media tra chi risponde          : {respondientes.mean():.2f}")

Ora confrontiamo due modi di ottenere un campione di dimensione *n*: **(a)** a caso tra tutti i clienti e **(b)** tra quelli che rispondono (distorto). Misuriamo l'errore assoluto medio della media campionaria ripetendo il processo 300 volte.

In [ ]:
filas = []
for n in [50, 500, 5_000, 20_000]:
    err_azar, err_sesgo = [], []
    for _ in range(300):
        err_azar.append(abs(rng.choice(satisfaccion, n, replace=False).mean() - mu_sat))
        err_sesgo.append(abs(rng.choice(respondientes, n, replace=False).mean() - mu_sat))
    filas.append({"n": n, "errore campione casuale": np.mean(err_azar), "errore campione distorto": np.mean(err_sesgo)})

sesgo = pd.DataFrame(filas).set_index("n").round(3)
display(sesgo)

sesgo.plot(marker="o", logx=True, figsize=(8, 4), title="Errore della media in base alla dimensione del campione")
plt.ylabel("Errore assoluto medio"); plt.show()

**Cosa osservi?**

- L'errore del campione **casuale** diminuisce al crescere di *n* (è solo rumore).
- L'errore del campione **distorto** resta fermo attorno allo stesso valore: quella è la distorsione. Con 20 000 risposte sbagli ancora quanto con 500.

**Conclusione:** la dimensione del campione riduce il rumore, ma non la distorsione. Prima di chiederti *"quanti dati ho?"*, chiediti *"come sono arrivati nella mia tabella e chi rappresentano?"*.

---
## Riepilogo della lezione

1. **Popolazione / campione / parametro / statistica campionaria:** la statistica campionaria stima il parametro e quasi mai coincide con esso.
2. **Variabilità campionaria:** ogni campione dà un valore diverso; l'oscillazione diminuisce come 1/√n (quattro volte più dati, metà del rumore).
3. **Tipo di variabile:** decide quale riepilogo ha senso; un numero non è sempre una variabile numerica.
4. **Segnale e rumore:** una differenza osservata in un campione può essere puro caso; le distribuzioni di "c'è un effetto" e "non c'è un effetto" si sovrappongono.
5. **Distorsione (bias) di campionamento:** più dati non la correggono; bisogna controllare come è stato ottenuto il campione.

**Prossimo passo:** Lezione 2 — Statistica descrittiva, in cui imparerai a riassumere un campione con media, mediana, dispersione e quantili. E ora, esercitati: `notebooks_practica/clase01_practica_pensar_estadisticamente_IT.ipynb`.